# Go/No-Go Analysis

Este notebook analiza los resultados de la tarea Go/No-Go, una tarea de **control inhibitorio**.

## La Tarea
- **Go trials (Azul)**: El participante debe presionar la barra espaciadora
- **No-Go trials (Naranja)**: El participante NO debe presionar nada

## Métricas Principales
- **HR (Hit Rate)**: Proporción de aciertos en Go trials
- **FA (False Alarm)**: Proporción de falsas alarmas en No-Go trials
- **c**: Criterio de respuesta (estandarizado)
- **sensibilidad**: Capacidad de discriminación
- **eficiencia**: Acc(Go) - Acc(NoGo)


In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.loader import get_latest_gonogo_analysis, load_gonogo_analysis

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)


## 1. Cargar Datos

### Opción A: Cargar análisis existente (rápido)


In [ ]:
# Cargar el análisis más reciente
result = get_latest_gonogo_analysis()

if result is not None:
    df, config_info = result
    print(f"Análisis cargado: {config_info.get('timestamp', 'N/A')}")
    print(f"Total sujetos: {len(df)}")
    print(f"  - Datapruebas: {len(df[df['origin'] == 'datapruebas'])}")
    print(f"  - Neuropruebas: {len(df[df['origin'] == 'neuropruebas'])}")
else:
    print("No se encontró análisis previo. Ejecuta la celda de Opción B.")


### Opción B: Ejecutar análisis fresco (más lento)


In [ ]:
# Descomentar para ejecutar análisis desde cero
# df, save_path = load_gonogo_analysis(save_results=True)
# print(f"Análisis completado. Guardado en: {save_path}")


## 2. Exploración de Datos


In [ ]:
df.head(10)


In [ ]:
df.describe()


## 3. HR vs FA (Signal Detection Theory)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ROC Space
ax1 = axes[0]
for origin in ['datapruebas', 'neuropruebas']:
    subset = df[df['origin'] == origin]
    ax1.scatter(subset['FA'], subset['HR'], alpha=0.6, label=origin, s=50)
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Chance')
ax1.scatter([0], [1], color='green', s=200, marker='*', label='Ideal', zorder=5)
ax1.set_xlabel('False Alarm Rate (FA)')
ax1.set_ylabel('Hit Rate (HR)')
ax1.set_title('ROC Space: HR vs FA')
ax1.legend()
ax1.set_xlim(-0.05, 1.05)
ax1.set_ylim(-0.05, 1.05)

# Distribuciones
ax2 = axes[1]
df[['HR', 'FA']].hist(ax=ax2, bins=20, alpha=0.7)
ax2.set_title('Distribución de HR y FA')

plt.tight_layout()
plt.show()

print(f"Correlación HR-FA: r = {df['HR'].corr(df['FA']):.3f}")


## 4. Criterio (c) y Sensibilidad


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Criterio c
ax1 = axes[0]
df['c'].dropna().hist(ax=ax1, bins=30, edgecolor='black', alpha=0.7)
ax1.axvline(0, color='red', linestyle='--', label='c=0 (neutral)')
ax1.set_xlabel('Criterio (c)')
ax1.set_ylabel('Frecuencia')
ax1.set_title(f'Criterio de Respuesta (μ={df["c"].mean():.2f})')
ax1.legend()

# Sensibilidad
ax2 = axes[1]
df['sensibilidad'].dropna().hist(ax=ax2, bins=30, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Sensibilidad')
ax2.set_ylabel('Frecuencia')
ax2.set_title(f'Sensibilidad (μ={df["sensibilidad"].mean():.2f})')

plt.tight_layout()
plt.show()

print(f"c > 0 (conservador): {(df['c'] > 0).sum()} sujetos")
print(f"c < 0 (liberal): {(df['c'] < 0).sum()} sujetos")


## 5. Comparación por Origen


In [ ]:
metrics = ['accuracy', 'HR', 'FA', 'c', 'sensibilidad', 'eficiencia']
comparison = df.groupby('origin')[metrics].agg(['mean', 'std', 'count'])
print("Comparación por origen:")
comparison.round(3)


## 6. Resumen Final


In [ ]:
print("="*60)
print("RESUMEN GO/NO-GO ANALYSIS")
print("="*60)
print(f"\nTotal sujetos analizados: {len(df)}")
print(f"  - Datapruebas: {len(df[df['origin'] == 'datapruebas'])}")
print(f"  - Neuropruebas: {len(df[df['origin'] == 'neuropruebas'])}")

print(f"\nMétricas principales:")
print(f"  Accuracy: {df['accuracy'].mean():.2f} ± {df['accuracy'].std():.2f}")
print(f"  Hit Rate (HR): {df['HR'].mean():.2f} ± {df['HR'].std():.2f}")
print(f"  False Alarm (FA): {df['FA'].mean():.2f} ± {df['FA'].std():.2f}")
print(f"  Criterio (c): {df['c'].mean():.2f} ± {df['c'].std():.2f}")
print(f"  Sensibilidad: {df['sensibilidad'].mean():.2f} ± {df['sensibilidad'].std():.2f}")
print(f"  Media RT: {df['media_rt'].mean():.0f} ± {df['media_rt'].std():.0f} ms")
